# Model Implementation

This Notebook explains in detail how to implement a neural network model into the Start Trek project and how to test it.

### Model class creation

This is a basic QNetwork structure. All the models we implemented are based on it.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class QNetwork(nn.Module):
    # State_size and action_size are given by the env,
    # seed is given by the user to set the random of the training
    # activation correspond to the activation function defined in the configuration file
    def __init__(self, state_size, action_size, seed, activation, hidden_layer_neuron=64, nb_layer=1): # Some models might take other parameter for __init__
        super(QNetwork, self).__init__()
        self.seed = torch.manual_seed(seed)
        
        self.activation = activation

        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(state_size, hidden_layer_neuron)) # create activation layer

        for _ in range(nb_layer):
            self.layers.append(nn.Linear(hidden_layer_neuron, hidden_layer_neuron)) # add [nb_layer] to the network with [hidden_layer_neuron] neurons

        self.output = nn.Linear(hidden_layer_neuron, action_size) # create output layer

    def forward(self, state):
        for layer in self.layers:
            state = self.activation(layer(state)) # apply activation function to the layers

        return self.output(state)

## Agent Class

The Agent class contains the methods called during the training.

In [ ]:
# IMPORT ===================================================
import numpy as np
import gymnasium as gym
import random
from collections import namedtuple, deque
from datetime import datetime, timedelta

from .model import QNetwork

import torch
import torch.nn.functional as F
import torch.optim as optim
import yaml
import os
from ..log_variable import device, RUNS_DIR, config_path, DATE_FORMAT, max_action
from ..utils.function_finders import get_activation, get_optimizer
#========================================================

class AgentDQN():
    from ..utils.plotting import save_graph, plot_scores, classify_termination, log_with_termination_stats, update_termination_stats

    def __init__(self, hyperparameter_set=None, hyperparameters=None, nb_episode=20000, seed=42):

        if hyperparameters is not None:
            self.hyperparameters = hyperparameters
            self.hyperparameter_set = "loaded_model"
        else:
            if not os.path.isfile(config_path): 
                print(f"File '{config_path}' not found")
                exit(84)

            with open(config_path, "r") as file:
                all_hyperparameter_sets = yaml.safe_load(file)

            if hyperparameter_set not in all_hyperparameter_sets:
                print(f"Model '{hyperparameter_set}' not found")
                exit(84)

            self.hyperparameter_set = hyperparameter_set
            self.hyperparameters = all_hyperparameter_sets[hyperparameter_set]

        self.nb_episode = nb_episode
        self.seed = seed
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        self.nb_step = 0

        hp = self.hyperparameters

        # === SHORTCUTS ===
        algo = hp["algo"]
        hypers = hp["hypers"]
        memory = hypers["memory"]
        epsilon_cfg = hypers["epsilon"]
        envs = hp["env"]
        winds = envs["wind"]
        gravity = envs["gravity"]
        fuel = envs["fuel"]
        continuous = envs["continuous"]

        # === Network ===
        self.nb_neuron = hp["fc1_nodes"]
        self.nb_layer = hp["number_of_layer"]

        # === Env ===
        self.enable_wind = winds["enable_wind"]
        self.wind_power = winds["wind_power"]
        self.turbulence_power = winds["turbulence_power"]

        self.gravity = gravity["gravity"]

        self.enable_fuel = fuel["enable_fuel"]

        self.enable_continuous = continuous["enable_continuous"]

        # === HYPERS ===
        self.replay_memory_size = memory["replay_memory_size"]
        self.batch_size = memory["mini_batch_size"]

        self.epsilon_init = epsilon_cfg["epsilon_init"]
        self.epsilon_decay = epsilon_cfg["epsilon_decay"]
        self.epsilon_min = epsilon_cfg["epsilon_min"]

        self.lr = hypers["learning_rate_a"]
        self.stop_on_reward = hypers["stop_on_mean_reward"]
        self.discount_factor_g = 0.99

        # === Algo ===
        self.activation_function = get_activation(algo["activation_function"])
        self.optimizer_function = get_optimizer(algo["optimizer"])


        # === Neural Network ===
        self.loss_fn = torch.nn.MSELoss()
        self.optimizer = None

        # === Path to Run info ===
        self.LOG_FILE = os.path.join(RUNS_DIR, f'{self.hyperparameter_set}-DQN.log')
        self.MODEL_FILE = os.path.join(RUNS_DIR, f'{self.hyperparameter_set}-DQN')
        self.GRAPH_FILE = os.path.join(RUNS_DIR, f'{self.hyperparameter_set}-DQN.png')

        self.termination_stats = {
            "landed": 0,
            "crash_high_speed": 0,
            "crash_bad_angle": 0,
            "out_of_bounds": 0,
            "time_limit": 0,
            "unknown": 0
        }

Each Agent __init__ function reads the configuration file given by the user and store the hyperparameters. 

(Following function are defined as method in the Agent class).

### Training setup

*_setup_training* is called once with the train script.
Only the creation of the neural network and memory object slightly differ with each models.
But the structure is always the same.

In [ ]:
def _setup_training(self, env):
        # We get the env specification
        self.action_size = env.action_space.n 
        self.state_size = env.observation_space.shape[0]

        # and we create the neural network based on the env information from above
        self.qnetwork_local = QNetwork(self.state_size, self.action_size, self.seed, self.activation_function, self.nb_neuron, self.nb_layer).to(device)
        self.qnetwork_target = QNetwork(self.state_size, self.action_size, self.seed, self.activation_function, self.nb_neuron, self.nb_layer).to(device)

        # We instantiate the memory object
        self.memory = ReplayBuffer(self.action_size, self.replay_memory_size, self.batch_size, self.seed)
        
        # The optimizer function is given in the configuration file
        self.optimizer = self.optimizer_function(self.qnetwork_local.parameters(), lr=self.lr)

### Logging setup

*_setup_logging* is called once with the train script.
This method is the same for all Agents. It writes a message when the training starts

In [ ]:
def _setup_logging(self, log_file):
        start_time = datetime.now()
        last_graph_update_time = start_time

        log_message = f"{start_time.strftime(DATE_FORMAT)}: Training starting..."
        print(log_message)
        log_file.write(log_message + '\n')
        log_file.flush()
        return last_graph_update_time, log_message

### Step

The *step* methode call the learn method every 4 steps. 

In [ ]:
def step(self, state, action, reward, next_state, done):
        self.memory.add(state, action, reward, next_state, done)
        
        step_every = 4
        discount_factor = 0.99

        self.nb_step = (self.nb_step + 1) % step_every
        if self.nb_step == 0:
            if len(self.memory) > self.batch_size:
                experiences = self.memory.sample()
                self.learn(experiences, discount_factor)

### learn

The *learn* methode is different for each model. But they all call the loss function and the *step* method to update the Network based on the training stored in memory.

In [ ]:
# Learn method for DQN Agent
def learn(self, experiences, gamma):
        states, actions, rewards, next_states, dones = experiences

        # Get max predicted Q values (for next states) from target model
        Q_targets_next = self.qnetwork_target(next_states).detach().max(1)[0].unsqueeze(1)
        # Compute Q targets for current states 
        Q_targets = rewards + (gamma * Q_targets_next * (1 - dones))

        # Get expected Q values from local model
        Q_expected = self.qnetwork_local(states).gather(1, actions)

        # Compute loss
        loss = F.mse_loss(Q_expected, Q_targets)
        # Minimize the loss
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # ------------------- update target network ------------------- #
        tau = 0.001
        self.soft_update(self.qnetwork_local, self.qnetwork_target, tau)

def soft_update(self, local_model, target_model, tau):
        for target_param, local_param in zip(target_model.parameters(), local_model.parameters()):
            target_param.data.copy_(tau*local_param.data + (1.0-tau)*target_param.data)

### Act

*act* method are very similar for all the models but certain ones wil need some modification in order to compel with the network model used.

In [ ]:
# Act method for DQN
def act(self, state, eps=0.):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        self.qnetwork_local.eval()
        with torch.no_grad():
            action_values = self.qnetwork_local(state)
        self.qnetwork_local.train()

        # Epsilon-greedy action selection
        if random.random() > eps:
            return np.argmax(action_values.cpu().data.numpy())
        else:
            return random.choice(np.arange(self.action_size))
        # Actions are more or less random based on the epsilon

### Training update

*_training_update* apply the epsilon decay and update a graph to see the model performances.

In [ ]:
def _training_update(self, rewards_per_episode,
                      epsilon_history, epsilon, last_graph_update_time):

        epsilon = max(epsilon * self.epsilon_decay, self.epsilon_min)
        epsilon_history.append(epsilon)

        current_time = datetime.now()
        if current_time - last_graph_update_time > timedelta(seconds=60):
            self.save_graph(rewards_per_episode, epsilon_history)
            last_graph_update_time = current_time

        return last_graph_update_time, epsilon

### Run

The *run* is the main method. It creates the env for the simulation and calls all the method above in a loop for a number of "episode" define by the user (see README).
This is also the method that cmpute the reward at the end of an episode to mesure the performances of the model.

In [ ]:
def run(self, visualization=False):
        #=====================================================================================
        # Log Message
        log_file = open(self.LOG_FILE, 'w') 
        last_graph_update_time, log_message = self._setup_logging(log_file)
        # Create instance of the environment and agent based on the configuration stored in the Agent class
        env = gym.make("LunarLander-v3", render_mode="human" if visualization else "rgb_array",
                        continuous=self.enable_continuous, gravity=self.gravity,
                        enable_wind=self.enable_wind,
                        wind_power=self.wind_power, turbulence_power=self.turbulence_power)
        env.reset(seed=self.seed)
        # Setup training value or load best model
        self._setup_training(env)
        #=====================================================================================

        # Train variable
        start_time = datetime.now()
        score = 0.0
        mean_rewards = []
        scores = []
        last_100_episode_mean = -999
        episode = 0
        epsilon_history = []
        eps = self.epsilon_init                    # initialize epsilon

        try:
            while (episode < self.nb_episode):
                state, _ = env.reset()
                score = 0
                for _ in range(max_action):
                    action = self.act(state, eps)
                    next_state, reward, terminated, truncated, _ = env.step(action)
                    done = terminated or truncated # done is use to detect the end of an episode, success like fail
                    self.step(state, action, reward, next_state, done)
                    # we update the state after the model has taken an action. We can say that the next_state correspond to the consequences of that action
                    state = next_state 
                    # finally we increment the score with the reward
                    score += reward
                    if done:
                        self.update_termination_stats(next_state, terminated, truncated)
                        break
                mean_rewards.append(score)
                scores.append(score)
                last_graph_update_time, eps = self._training_update(mean_rewards, epsilon_history, eps, last_graph_update_time)


                episode += 1
                if len(mean_rewards) >= 100:
                    last_100_episode_mean = np.mean(mean_rewards[-100:])

                if episode % 100 == 0: # Print log and save them in a text file
                    self.log_with_termination_stats(
                        log_file,
                        episode,
                        last_100_episode_mean,
                        eps,
                        last_graph_update_time,
                        start_time
                    )
                # The training stops when the mean reward is superior to the stop_on_reward defined in the configuration file
                if last_100_episode_mean >= self.stop_on_reward:
                    self.log_with_termination_stats(
                        log_file,
                        episode,
                        last_100_episode_mean,
                        eps,
                        last_graph_update_time,
                        start_time
                    )
                    log_message = f"Solved in {episode} episodes! With a score of {np.mean(mean_rewards[-100:])}"
                    print(log_message)
                    log_file.write(log_message + '\n')
                    log_file.flush()
                    torch.save({ 
                        "model_state_dict": self.qnetwork_local.state_dict(),
                        "hyperparameters": self.hyperparameters,
                    }, self.MODEL_FILE + "_solved.pt")
                    break
        except KeyboardInterrupt:
            torch.save({
                    "model_state_dict": self.qnetwork_local.state_dict(),
                    "hyperparameters": self.hyperparameters,
                }, self.MODEL_FILE + "_last.pt")
        AgentDQN.plot_scores(scores, self.MODEL_FILE + "_detail.png")
        return mean_rewards

## Update Training script

You just need to add the new agent in the lists of models already implemented.

In [ ]:
from codebase.AgentRainbowDQN.RainbowDQN import AgentRainbowDQN
from codebase.AgentDDQN_PER_Dueling_N_Step_Noisy.DDQN_PER_Dueling_N_Step_Noisy import AgentDDQN_PER_Dueling_N_Step_Noisy
from codebase.AgentDDQN_PER_Dueling_N_Step.DDQN_PER_Dueling_N_Step import AgentDDQN_PER_Dueling_N_Step
from codebase.AgentDDQN_PER_Dueling.DDQN_PER_Dueling import AgentDDQN_PER_Dueling
from codebase.AgentDDQN_PER.DDQN_PER import AgentDDQN_PER
from codebase.AgentDDQN.DDQN import AgentDDQN
from codebase.AgentDQN.DQN import AgentDQN
from codebase.AgentPPO.PPO import AgentPPO
# import agent class declaration file 
import sys

def choose_model():
    print("\n==============================")
    print("   RL Model Selection")
    print("==============================\n")
    
    print("1. DQN")
    print("2. DDQN")
    print("3. DDQN + PER")
    print("4. DDQN + PER + Dueling")
    print("5. DDQN + PER + Dueling + N-Step")
    print("6. DDQN + PER + Dueling + N-Step + Noisy Net")
    print("7. Rainbow DQN")
    print("8. PPO\n")
    # add new option

    while True:
        try:
            choice = int(input("-> Enter your choice (1-8): "))
            if 1 <= choice <= 8: # Update condition to take more options
                break
            else:
                print("Invalid choice, please enter a number between 1 and 8.")
        except ValueError:
            print("Please enter a valid number.")

    models = {
        1: "DQN",
        2: "DDQN",
        3: "DDQN + PER",
        4: "DDQN + PER + Dueling",
        5: "DDQN + PER + Dueling + N-Step",
        6: "DDQN + PER + Dueling + N-Step + Noisy Net",
        7: "Rainbow DQN",
        8: "PPO"
        # also add the new option here
    }

    print(f"\nYou selected: {models[choice]}\n")
    return choice


# shell or add an argmument
if __name__ == '__main__':
    models = {
        1: lambda nb_episode, seed: AgentDQN(
                                        hyperparameter_set=sys.argv[1],
                                        nb_episode=nb_episode if nb_episode > 0 else 20000,
                                        seed=seed if seed >= 0 else 42),
        2: lambda nb_episode, seed: AgentDDQN(
                                        hyperparameter_set=sys.argv[1],
                                        nb_episode=nb_episode if nb_episode > 0 else 20000,
                                        seed=seed if seed >= 0 else 42),
        3: lambda nb_episode, seed: AgentDDQN_PER(
                                        hyperparameter_set=sys.argv[1],
                                        nb_episode=nb_episode if nb_episode > 0 else 20000,
                                        seed=seed if seed >= 0 else 42),
        4: lambda nb_episode, seed: AgentDDQN_PER_Dueling(
                                        hyperparameter_set=sys.argv[1],
                                        nb_episode=nb_episode if nb_episode > 0 else 20000,
                                        seed=seed if seed >= 0 else 42),
        5: lambda nb_episode, seed: AgentDDQN_PER_Dueling_N_Step(
                                        hyperparameter_set=sys.argv[1],
                                        nb_episode=nb_episode if nb_episode > 0 else 20000,
                                        seed=seed if seed >= 0 else 42),
        6: lambda nb_episode, seed: AgentDDQN_PER_Dueling_N_Step_Noisy(
                                        hyperparameter_set=sys.argv[1],
                                        nb_episode=nb_episode if nb_episode > 0 else 20000,
                                        seed=seed if seed >= 0 else 42),
        7: lambda nb_episode, seed: AgentRainbowDQN(
                                        hyperparameter_set=sys.argv[1],
                                        nb_episode=nb_episode if nb_episode > 0 else 20000,
                                        seed=seed if seed >= 0 else 42),
        8: lambda nb_episode, seed: AgentPPO(
                                        hyperparameter_set=sys.argv[1],
                                        nb_episode=nb_episode if nb_episode > 0 else 20000,
                                        seed=seed if seed >= 0 else 42),
                                    }
        # Create a lambda function that creates the new agent

    if len(sys.argv) != 4:
        print("Not enough argument : python3 -m codebase.train <model_name> <nb_episode> <seed>")
        exit(84)

    try:
        nb_episode = int(sys.argv[2])
        seed = int(sys.argv[3])
    except ValueError:
        print("Usage : python3 -m codebase.train <model_name> <nb_episode> <seed>")
        print("<nb_episode> and <seed> must be valid number")
        exit(84)

    choice = choose_model()
    model = models[choice](nb_episode, seed)

    # model.run(visualization=True)
    # model.run()

## Update Evaluation script

Similar to the Training, you just need to add in the option the new model implemented.
Some other specific modification might be needed for certain models.

In [ ]:
import os
import sys
import torch
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

from .utils.function_finders import get_activation, get_optimizer

from codebase.AgentDQN.model import QNetwork as DQN_Network
from codebase.AgentDDQN.model import QNetwork as DDQN_Network
from codebase.AgentDDQN_PER.model import QNetwork as DDQN_PER_Network
from codebase.AgentDDQN_PER_Dueling.model import QNetwork as DDQN_PER_Dueling_Network
from codebase.AgentDDQN_PER_Dueling_N_Step.model import QNetwork as DDQN_PER_Dueling_N_Step_Network
from codebase.AgentDDQN_PER_Dueling_N_Step_Noisy.model import QNetwork as DDQN_PER_Dueling_N_Step_Noisy_Network
from codebase.AgentRainbowDQN.model import QNetwork as Rainbow_Network
from codebase.AgentPPO.model import ActorNetwork as PPO_Network
# import new model when implemented

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def evaluate_all_models(runs_dir="runs", nb_episode=100, seed=42):
    results = {}

    np.random.seed(seed)
    torch.manual_seed(seed)

    for file in os.listdir(runs_dir):
        if not file.endswith(".pt"):
            continue

        print(f"\nEvaluating {file}")

        checkpoint = torch.load(os.path.join(runs_dir, file), map_location=device)

        if "model_state_dict" not in checkpoint or "hyperparameters" not in checkpoint:
            print(f"Skipping {file} (invalid checkpoint)")
            continue

        config = checkpoint["hyperparameters"]
        env = config["env"]
        wind = env["wind"]
        gravity = env["gravity"]

        env = gym.make("LunarLander-v3",
                    render_mode="rgb_array",
                    enable_wind=wind["enable_wind"],
                    wind_power=wind["wind_power"],
                    turbulence_power=wind["turbulence_power"],
                    gravity=gravity["gravity"]
        )

        model_name = file.replace(".pt", "")
        video_path = os.path.join(runs_dir, "videos", model_name)

        env = RecordVideo(
            env,
            video_folder=video_path,
            episode_trigger=lambda ep: ep > nb_episode - 4,  # record last 3 episodes
            name_prefix=model_name
        )

        env.action_space.seed(seed)

        state_dim = env.observation_space.shape[0]
        action_dim = env.action_space.n

        MODEL_MAP = [
            ("Rainbow", Rainbow_Network),
            ("DDQN_PER_Dueling_N_Step_Noisy", DDQN_PER_Dueling_N_Step_Noisy_Network),
            ("DDQN_PER_Dueling_N_Step", DDQN_PER_Dueling_N_Step_Network),
            ("DDQN_PER_Dueling", DDQN_PER_Dueling_Network),
            ("DDQN_PER", DDQN_PER_Network),
            ("DDQN", DDQN_Network),
            ("DQN", DQN_Network),
            ("PPO", PPO_Network),
            # Add new model
        ]

        model = None
        for key, ModelClass in MODEL_MAP:
            if key in file:
                if key == "PPO":
                    model = ModelClass(
                        n_actions=action_dim,
                        input_dims=state_dim,
                        alpha=0.0,  # not used in eval
                        activation=get_activation(config["algo"]["activation_function"]),
                        optimizer=torch.optim.Adam, # not used in eval
                        hidden_dim=config["fc1_nodes"],
                        nb_layer=config["number_of_layer"]
                    ).to(device)
                # Some model might need a different initialization so put a "elif" if needed
                else:
                    model = ModelClass(
                        state_size=state_dim,
                        action_size=action_dim,
                        seed=seed,
                        activation=get_activation(config["algo"]["activation_function"]),
                        hidden_layer_neuron=config["fc1_nodes"],
                        nb_layer=config["number_of_layer"]
                    ).to(device)
                break

        if "PPO" in file:
            model.load_state_dict(checkpoint["model_state_dict"]["actor"])
        # Some models have a special state_dict so put a "elif" if needed
        else:
            model.load_state_dict(checkpoint["model_state_dict"])
        model.eval()

        rewards = []

        # --- Evaluation loop ---
        for episode in range(nb_episode):
            state, _ = env.reset(seed=seed + episode)
            state = torch.from_numpy(state).float().to(device)

            episode_reward = 0.0

            with torch.no_grad():
                for step in range(1000):
                    if "PPO" in file: # PPO return Categorical instead of just an output
                        dist = model(state.unsqueeze(0))
                        action = torch.argmax(dist.probs, dim=1).item()
                    elif "Rainbow" in file: # atoms 
                        dist = model(state.unsqueeze(0))
                        support = torch.linspace(-300, 300, 51).to(device)
                        expected_q = (dist * support).sum(dim=2)
                        action = expected_q.argmax(dim=1).item()
                    # Some models do not take actions as the other and need another call so put a "elif" if needed
                    else:
                        action = model(state.unsqueeze(0)).argmax(dim=1).item()

                    next_state, reward, terminated, truncated, _ = env.step(action)
                    done = terminated or truncated

                    episode_reward += reward
                    state = torch.from_numpy(next_state).float().to(device)

                    if done:
                        break

            rewards.append(episode_reward)

        # --- Metrics ---
        rewards = np.array(rewards)

        results[file] = {
            "mean": float(np.mean(rewards)),
            "std": float(np.std(rewards)),
            "median": float(np.median(rewards)),
            "min": float(np.min(rewards)),
            "max": float(np.max(rewards)),
            "success_rate": float(np.mean(rewards >= 200))
        }

        env.close()

    # ================= RESULTS =================
    print("\n=== RESULTS ===")
    for model_name, metrics in results.items():
        print(model_name, metrics)

    # --- Plot ---
    sorted_items = sorted(results.items(), key=lambda x: x[1]["mean"], reverse=True)

    model_files = [k for k, _ in sorted_items]
    model_names = [k.replace(".pt", "") for k in model_files]
    model_names = [name.replace(".pt", "").replace("LunarLander-", "") for name in model_files]

    means = [v["mean"] for _, v in sorted_items]
    stds = [v["std"] for _, v in sorted_items]
    success_rates = [v["success_rate"] for _, v in sorted_items]

    plt.figure(figsize=(25, 5))

    x = np.arange(len(model_names))

    plt.subplot(1, 3, 1)
    plt.bar(x, means, width=0.6)
    plt.title("Mean Reward")
    plt.xticks(x, model_names, rotation=45, ha='right', fontsize=8)

    plt.subplot(1, 3, 2)
    plt.bar(x, stds, width=0.6)
    plt.title("Std Dev")
    plt.xticks(x, model_names, rotation=45, ha='right', fontsize=8)

    plt.subplot(1, 3, 3)
    plt.bar(x, success_rates, width=0.6)
    plt.title("Success Rate")
    plt.xticks(x, model_names, rotation=45, ha='right', fontsize=8)

    plt.tight_layout()
    plt.savefig(os.path.join(runs_dir, "comparison.png"))
    plt.close()

    # --- CSV ---
    df = pd.DataFrame(results).T.reset_index()
    df.columns = ["model", "mean", "std", "median", "min", "max", "success_rate"]

    df = df.sort_values("mean", ascending=False)
    df.to_csv(os.path.join(runs_dir, "results.csv"), index=False)

    return results


This is all you need to implement a neural network model in our Start Trek project.